# Aula 2 (assíncrona) - Parte A: notebook guiado

**Grandes Modelos de Linguagem · Residência em Engenharia de Software Inteligente (FACOM/UFMS)**
Prof. Bruno Nogueira · período: 05/11 a 18/11/2026 · atividade **individual** · cerca de 1h30

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmnogueira-ufms/LLM-ESIA-2026/blob/main/aula02/aula02_notebook_guiado.ipynb)

Neste notebook você vai colocar a mão na massa com os conceitos da Aula 1:

| Exercício | Família / conceito | O que você vai fazer |
|---|---|---|
| 1 | Tokenização | comparar três tokenizadores em textos do **seu MVP** |
| 2 | Encoder-only | gerar embeddings com o BERTimbau, medir similaridade e treinar um classificador |
| 3 | Encoder-decoder | resumir um texto com um T5 em português |
| 4 | Decoder-only | fazer sua primeira chamada a um LLM e resolver **a mesma classificação** do exercício 2 em *zero-shot* |

As células marcadas com **✍️** pedem uma resposta curta sua. Elas vão alimentar a **ficha do caso de uso** (Parte B, em grupo).

> **Antes de começar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.

## 0. Preparação

In [ ]:
!pip install -q tiktoken openai

In [ ]:
import time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import tiktoken
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)

### Seus textos

Substitua os exemplos abaixo por **5 a 10 textos reais (ou realistas) do domínio do seu MVP** - mensagens de usuários, descrições, documentos curtos. **Não use dados pessoais ou sensíveis.**

In [ ]:
TEXTOS_DO_MVP = [
    "Não consigo fazer login no portal desde ontem, aparece erro de senha inválida.",
    "Preciso de uma declaração de matrícula para apresentar no estágio.",
    "Fui cobrado duas vezes pela taxa de emissão do diploma.",
    # TODO: substitua por textos do seu MVP
]

---
## Exercício 1 - Tokenização

Compare três tokenizadores de famílias diferentes:
- **BERTimbau** (`neuralmind/bert-base-portuguese-cased`) - encoder-only, WordPiece, treinado em português;
- **GPT-4o** (`o200k_base`, via `tiktoken`) - decoder-only proprietário, BPE em bytes;
- **Qwen 2.5** (`Qwen/Qwen2.5-1.5B-Instruct`) - decoder-only aberto, BPE em bytes.

In [ ]:
tok_bert = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
tok_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
enc_gpt4o = tiktoken.get_encoding("o200k_base")

def contar(texto):
    return {
        "BERTimbau": len(tok_bert(texto, add_special_tokens=False)["input_ids"]),
        "GPT-4o":    len(enc_gpt4o.encode(texto)),
        "Qwen2.5":   len(tok_qwen(texto, add_special_tokens=False)["input_ids"]),
    }

linhas = []
for t in TEXTOS_DO_MVP:
    c = contar(t)
    c["palavras"] = len(t.split())
    c["texto"] = t[:60] + ("…" if len(t) > 60 else "")
    linhas.append(c)
df_tok = pd.DataFrame(linhas)
df_tok

In [ ]:
# Tokens por palavra, em média, para cada tokenizador
media = {nome: (df_tok[nome] / df_tok["palavras"]).mean() for nome in ["BERTimbau", "GPT-4o", "Qwen2.5"]}
print({k: round(v, 2) for k, v in media.items()})

plt.bar(media.keys(), media.values(), color=["#1E9E8F", "#F0643C", "#7B61FF"])
plt.ylabel("tokens por palavra")
plt.title("Textos do meu MVP")
plt.show()

In [ ]:
# TODO: escolha o texto mais "difícil" da sua lista (siglas, nomes próprios, gírias, códigos)
# e veja como cada tokenizador o quebra.
texto = TEXTOS_DO_MVP[0]
print("BERTimbau:", tok_bert.tokenize(texto))
print("GPT-4o:   ", [enc_gpt4o.decode([i]) for i in enc_gpt4o.encode(texto)])
print("Qwen2.5:  ", [tok_qwen.decode([i]) for i in tok_qwen(texto, add_special_tokens=False)["input_ids"]])

✍️ **Responda:** (a) Qual tokenizador foi mais econômico para os seus textos? (b) Quantos tokens tem, em média, uma entrada típica do seu MVP? (c) Algum termo importante do seu domínio foi muito fragmentado?

✍️ **Sua resposta:** _escreva aqui_

---
## Exercício 2 - Encoder-only: embeddings e classificação com o BERTimbau

### 2.1 Similaridade semântica

Para ter **um vetor por frase**, tiramos a média dos vetores de todos os tokens (*mean pooling*), ignorando o *padding*.

In [ ]:
bert = AutoModel.from_pretrained("neuralmind/bert-base-portuguese-cased").to(device).eval()

def embeddings(textos, batch=16):
    vetores = []
    for i in range(0, len(textos), batch):
        enc = tok_bert(textos[i:i + batch], padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            h = bert(**enc).last_hidden_state                  # (lote, tokens, 768)
        m = enc["attention_mask"].unsqueeze(-1).float()       # ignora [PAD]
        vetores.append(((h * m).sum(1) / m.sum(1)).cpu())
    v = torch.cat(vetores)
    return torch.nn.functional.normalize(v, dim=1).numpy()   # normalizado: produto interno = cosseno

frases = [
    "Esqueci minha senha do portal.",
    "Não consigo entrar no sistema.",
    "Preciso pagar a taxa do diploma.",
    "O boleto venceu ontem.",
]
E = embeddings(frases)
pd.DataFrame(E @ E.T, index=frases, columns=[f"f{i+1}" for i in range(len(frases))]).round(3)

> **Atenção:** o BERTimbau puro **não** foi treinado para similaridade de frases, então as similaridades tendem a ser todas altas. O que importa é a **ordem**: pares com o mesmo assunto devem ficar acima dos demais. Modelos treinados para isso (por exemplo, da biblioteca `sentence-transformers`) separam melhor - vamos usá-los no RAG da Aula 3.

In [ ]:
# TODO: calcule a similaridade entre os textos do SEU MVP. Os pares mais parecidos fazem sentido?
E_mvp = embeddings(TEXTOS_DO_MVP)
pd.DataFrame(E_mvp @ E_mvp.T).round(3)

### 2.2 Um classificador com embeddings

Vamos usar um conjunto de exemplo com **60 chamados fictícios** de uma secretaria acadêmica, em quatro categorias. Os embeddings do BERTimbau viram as *features* de uma regressão logística.

> Se o seu MVP tem uma tarefa de classificação, troque pelo seu CSV (colunas `texto` e `categoria`).

In [ ]:
URL = "https://raw.githubusercontent.com/bmnogueira-ufms/LLM-ESIA-2026/main/dados/chamados_exemplo.csv"
df = pd.read_csv(URL)
print(df["categoria"].value_counts())
df.sample(5, random_state=0)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

treino, teste = train_test_split(df, test_size=20, stratify=df["categoria"], random_state=42)

inicio = time.time()
X_treino, X_teste = embeddings(treino["texto"].tolist()), embeddings(teste["texto"].tolist())
clf = LogisticRegression(max_iter=2000).fit(X_treino, treino["categoria"])
pred_bert = clf.predict(X_teste)
tempo_bert = (time.time() - inicio) / len(df)

acc_bert = accuracy_score(teste["categoria"], pred_bert)
print(f"Acurácia BERTimbau + regressão logística: {acc_bert:.0%}  (tempo médio por exemplo: {1000 * tempo_bert:.0f} ms)")
print(classification_report(teste["categoria"], pred_bert, zero_division=0))

✍️ **Responda:** Quantos exemplos rotulados foram usados no treino? Em quais categorias o classificador mais errou? Você acha que esse desempenho seria aceitável no seu MVP?

✍️ **Sua resposta:** _escreva aqui_

---
## Exercício 3 - Encoder-decoder: sumarização com um T5 em português

Modelo: `recogna-nlp/ptt5-base-summ-xlsum` - um PTT5 (T5 pré-treinado em português) ajustado para resumir notícias.

> ⚠️ Verifique sempre a **licença** do modelo. Este é distribuído sob **CC BY-NC-SA 4.0**, que **não permite uso comercial**. Isso é uma restrição real para um MVP!

In [ ]:
from transformers import AutoModelForSeq2SeqLM

tok_t5 = AutoTokenizer.from_pretrained("recogna-nlp/ptt5-base-summ-xlsum")
t5 = AutoModelForSeq2SeqLM.from_pretrained("recogna-nlp/ptt5-base-summ-xlsum").to(device)

def resumir(texto, max_new_tokens=60):
    enc = tok_t5(texto, max_length=512, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = t5.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4, no_repeat_ngram_size=3)
    return tok_t5.decode(out[0], skip_special_tokens=True)

noticia = (
    "A prefeitura de uma capital do Centro-Oeste anunciou nesta segunda-feira um programa para ampliar o acesso à internet "
    "em escolas públicas. Segundo a secretaria de educação, 120 escolas receberão novos pontos de acesso sem fio até o fim "
    "do ano, e os professores participarão de oficinas sobre o uso de ferramentas digitais em sala de aula. O investimento "
    "previsto é de 8 milhões de reais, com recursos municipais e federais."
)   # texto fictício
print(resumir(noticia))

In [ ]:
# TODO: teste com um texto do seu domínio (de preferência com 5 a 10 linhas)
# print(resumir("..."))

✍️ **Responda:** O resumo manteve as informações principais? Inventou algo que não estava no texto (alucinação)? Uma tarefa de "transformar texto" aparece no seu MVP?

✍️ **Sua resposta:** _escreva aqui_

---
## Exercício 4 - Decoder-only: sua primeira chamada a um LLM

Agora vamos resolver **a mesma classificação do Exercício 2**, mas sem treinar nada: só com instruções no prompt (*zero-shot*). Escolha **uma** das opções:

- **Opção A - modelo aberto no próprio Colab** (sem chave, sem custo): `Qwen/Qwen2.5-1.5B-Instruct`. Precisa da GPU T4.
- **Opção B - API compatível com a OpenAI** (OpenRouter ou outro provedor): exige uma chave. Guarde a chave nos **Secrets do Colab** (ícone 🔑 na barra lateral) com o nome `LLM_API_KEY`. **Nunca** cole a chave no notebook nem a envie ao GitHub.

> Escolha em `OPCAO` abaixo e rode as células.

In [ ]:
OPCAO = "A"     # "A" = modelo local no Colab · "B" = API

# Configuração da opção B (ignore se usar a opção A)
BASE_URL   = "https://openrouter.ai/api/v1"   # ou o endpoint do seu provedor
MODELO_API = ""                               # TODO: nome do modelo no provedor escolhido

In [ ]:
if OPCAO == "A":
    from transformers import AutoModelForCausalLM
    llm = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device).eval()

    def chamar_llm(mensagens, max_new_tokens=20):
        enc = tok_qwen.apply_chat_template(mensagens, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(device)
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        n_in, n_out = enc["input_ids"].shape[1], out.shape[1] - enc["input_ids"].shape[1]
        return tok_qwen.decode(out[0, n_in:], skip_special_tokens=True), n_in, n_out
else:
    from openai import OpenAI
    from google.colab import userdata
    assert MODELO_API, "Preencha MODELO_API com o nome do modelo do seu provedor."
    cliente = OpenAI(base_url=BASE_URL, api_key=userdata.get("LLM_API_KEY"))

    def chamar_llm(mensagens, max_new_tokens=20):
        r = cliente.chat.completions.create(model=MODELO_API, messages=mensagens,
                                            max_tokens=max_new_tokens, temperature=0)
        uso = r.usage
        return r.choices[0].message.content, uso.prompt_tokens, uso.completion_tokens

resposta, n_in, n_out = chamar_llm([{"role": "user", "content": "Em uma frase: o que é um modelo de linguagem?"}], max_new_tokens=80)
print(resposta, f"\n\n(tokens de entrada: {n_in} · tokens de saída: {n_out})")

### 4.1 Classificação zero-shot

Repare na estrutura do prompt: **papel**, **tarefa**, **categorias possíveis** e **formato da resposta**. Na Aula 3 vamos aprofundar essas técnicas.

In [ ]:
CATEGORIAS = sorted(df["categoria"].unique())

SISTEMA = (
    "Você é um assistente da secretaria acadêmica de uma universidade. "
    "Sua tarefa é classificar chamados de estudantes."
)

def montar_prompt(texto):
    return [
        {"role": "system", "content": SISTEMA},
        {"role": "user", "content":
            "Classifique o chamado em exatamente uma das categorias: " + ", ".join(CATEGORIAS) + ".\n"
            "Responda apenas com o nome da categoria, sem explicações.\n\n"
            f"Chamado: {texto}"},
    ]

def normalizar(s):
    s = unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z_ ]", "", s).replace(" ", "_")

def extrair_categoria(resposta):
    r = normalizar(resposta)
    for c in CATEGORIAS:
        if all(parte in r for parte in c.split("_")):   # aceita "acesso_sistemas" ou "acesso a sistemas"
            return c
    return "invalida"          # o modelo fugiu do formato

print(montar_prompt(teste["texto"].iloc[0])[1]["content"])

In [ ]:
resultados = []
for texto, rotulo in zip(teste["texto"], teste["categoria"]):
    inicio = time.time()
    resposta, n_in, n_out = chamar_llm(montar_prompt(texto))
    resultados.append({"texto": texto, "esperado": rotulo, "resposta_bruta": resposta,
                       "previsto": extrair_categoria(resposta), "latencia_s": time.time() - inicio,
                       "tokens_in": n_in, "tokens_out": n_out})
res = pd.DataFrame(resultados)

acc_llm = (res["previsto"] == res["esperado"]).mean()
print(f"Acurácia LLM zero-shot: {acc_llm:.0%}")
print(f"Respostas fora do formato: {(res['previsto'] == 'invalida').sum()}")
print(f"Latência média: {res['latencia_s'].mean():.2f} s · tokens médios: {res['tokens_in'].mean():.0f} entrada / {res['tokens_out'].mean():.0f} saída")
res[res["previsto"] != res["esperado"]][["texto", "esperado", "resposta_bruta"]]

### 4.2 Comparando as duas abordagens

In [ ]:
pd.DataFrame([
    {"abordagem": "BERTimbau + regressão logística", "acurácia": f"{acc_bert:.0%}",
     "exemplos rotulados usados": len(treino), "tempo médio por exemplo": f"{1000 * tempo_bert:.0f} ms"},
    {"abordagem": f"LLM zero-shot (opção {OPCAO})", "acurácia": f"{acc_llm:.0%}",
     "exemplos rotulados usados": 0, "tempo médio por exemplo": f"{1000 * res['latencia_s'].mean():.0f} ms"},
])

✍️ **Responda:**
1. Qual abordagem teve melhor acurácia? E menor latência?
2. Olhe os erros do LLM: são erros "razoáveis" (casos ambíguos) ou absurdos? Houve resposta fora do formato?
3. Se este fosse o seu MVP, qual abordagem você escolheria hoje - e o que mudaria se você **não** tivesse nenhum dado rotulado?
4. (Opcional) Mude o prompt - por exemplo, incluindo uma frase de descrição para cada categoria - e veja se a acurácia muda.

✍️ **Sua resposta:** _escreva aqui_

---
## ✅ Antes de sair

- [ ] Rodei os quatro exercícios e respondi às perguntas ✍️.
- [ ] Anotei quantos tokens tem uma entrada típica do meu MVP (Exercício 1).
- [ ] Anotei o resultado da comparação do Exercício 4.
- [ ] Levei essas informações para o meu grupo - elas entram na **ficha do caso de uso** (Parte B), com entrega até **18/11**.